# Targeted accentuations across models

This notebook visualizes `PullbackAscentClipLast` for either one selected ImageNet target class or the ground-truth label of each image. The final figure shows the original images and the PVT, ResNet, and VGG accentuations. Set `MODEL_LAYOUT = "columns"` for one column per model (original images in the first column), or `"rows"` for one row per model (original images in the top row).

Set `target_class`, `K`, `alpha`, and `MODEL_LAYOUT` in the configuration cell, then run all cells. The final cell both displays the figure and saves it under `media/targeted_accentuations/`.
To change only the layout after computing the results, update `MODEL_LAYOUT` and rerun the final plotting cell; model computations do not need to be repeated.

Set `TARGET_MODE = "fixed"` to use `target_class` for every image, or `TARGET_MODE = "true_label"` to use each image's dataset label. After changing the target mode, rerun the configuration and subsequent cells to recompute accentuations.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import ast
import gc
import os
import re
from pathlib import Path

# Allow deterministic gradients on CUDA.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Make paths work whether the kernel starts in the repository root or notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display

from lib.defaults import get_default_kwargs
from lib.evaluator import QuantusEvaluator, default_explainers
from lib.helpers import plot_example_grid
from lib.setup import setup_dataloader, setup_device, setup_model
from lib.surrogates import soften_module_inplace_

deterministic = False  # Some model layers do not support deterministic gradients.
torch.use_deterministic_algorithms(deterministic)
torch.backends.cudnn.deterministic = deterministic

c:\Users\Maciej\Documents\GitHub\SemanticPullbacks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Main experiment parameters.
METHOD = "PullbackAscentClipLast"
# METHOD = "PullbackAscentClip"

# "fixed": shared target_class; "true_label": each image's dataset label.
TARGET_MODE = "fixed"

K = 10
alpha = 20
target_class = 497  # ImageNet class id; 497 = church.
# target_class = 0
# target_class = 8
target_class = 9
# target_class = 10
# target_class = 18
# target_class = 24
# target_class = 566
# target_class = 217
# target_class = 701
# target_class = 340
# target_class = 571
# target_class = 491

batch_size = 50
seed = 314159265359
# seed = 314
# seed = 42
seed = 2

N_EXAMPLES = 5

dataset = "imagenet"
# dataset = "imagenette"

# Display models in "rows" or "columns" in the final figure.
MODEL_LAYOUT = "columns"

# Model order in either layout.
MODEL_CONFIGS = [
    ("PVT", "pvt_v2_b1", "timm"),
    ("ResNet", "resnet50", "torchvision"),
    ("VGG", "vgg11_bn", "torchvision"),
]

if TARGET_MODE not in {"fixed", "true_label"}:
    raise ValueError('TARGET_MODE must be "fixed" or "true_label".')
if TARGET_MODE == "fixed" and not 0 <= target_class < 1000:
    raise ValueError("target_class must be an ImageNet class id in [0, 999].")
if K < 1:
    raise ValueError("K must be at least 1.")

In [ ]:
device = setup_device()
loader = setup_dataloader(
    batch_size=batch_size,
    seed=seed,
    dataset=dataset,
    size="full",
)

it = iter(loader)
images, labels = next(it)
images, labels = images[:N_EXAMPLES], labels[:N_EXAMPLES]
if TARGET_MODE == "true_label":
    targets = labels.detach().clone().to(dtype=torch.long)
    target_description = "ground-truth labels"
    target_file_tag = "true_label"
elif TARGET_MODE == "fixed":
    targets = torch.full((len(images),), target_class, dtype=torch.long)
else:
    raise ValueError('TARGET_MODE must be "fixed" or "true_label".')

with open("imagenet1000_clsidx_to_labels.txt", encoding="utf-8") as label_file:
    imagenet_labels = ast.literal_eval(label_file.read())
target_names = [imagenet_labels[int(label)].split(",")[0] for label in targets]
if TARGET_MODE == "fixed":
    target_name = target_names[0]
    target_description = f"{target_class} ({target_name})"
    target_slug = re.sub(r"[^a-zA-Z0-9_-]+", "_", target_name).strip("_").lower()
    target_file_tag = f"{target_class}_{target_slug}"

print(f"device: {device}")
print(f"target: {target_description}")
print("per-image targets:", list(zip(targets.tolist(), target_names)))
print(f"images: {tuple(images.shape)}")
plot_example_grid(
    images,
    nrow=len(images),
    title="Source images",
    dpi=80,
    fontsize=18,
)

In [4]:
temperatures, default_pga_kwargs, _, _ = get_default_kwargs()
pga_kwargs = {
    **default_pga_kwargs,
    "alpha": alpha,
    "steps": K,
}

def compute_targeted_accentuations(model_name, model_source):
    model = setup_model(
        device=device,
        model_name=model_name,
        model_source=model_source,
    )

    # Replace supported modules with their softened counterparts while keeping
    # ordinary backward behavior until the pullback explainer is called.
    soften_module_inplace_(
        model,
        temperatures=temperatures,
        standard_backward=True,
        fill_default_temperatures=True,
    )

    explainers = default_explainers(
        [METHOD],
        pga_kwargs_counterfactual=pga_kwargs,
    )
    evaluator = QuantusEvaluator(
        model=model,
        metrics={},
        explainers=explainers,
        device=device,
    )
    evaluator.precompute_attributions(images, targets)
    attributions = torch.as_tensor(evaluator.attributions[METHOD]).cpu()

    del evaluator, explainers, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return attributions

In [ ]:
model_attributions = {}

for row_name, model_name, model_source in MODEL_CONFIGS:
    print(f"Computing {row_name} ({model_name})...")
    model_attributions[row_name] = compute_targeted_accentuations(
        model_name=model_name,
        model_source=model_source,
    )

{name: tuple(attrs.shape) for name, attrs in model_attributions.items()}

In [ ]:
if MODEL_LAYOUT not in {"rows", "columns"}:
    raise ValueError('MODEL_LAYOUT must be "rows" or "columns".')

model_names = [name for name, _, _ in MODEL_CONFIGS]
group_titles = ["Images"] + model_names
image_groups = [images.cpu()] + [model_attributions[name] for name in model_names]

if MODEL_LAYOUT == "rows":
    figure_batch = torch.cat(image_groups, dim=0)
    grid_columns = len(images)
    grid_rows = len(image_groups)
    row_titles = group_titles
    column_titles = None
else:
    # Interleave originals and model results for each source image.
    figure_batch = torch.stack(image_groups, dim=1).flatten(0, 1)
    grid_columns = len(image_groups)
    grid_rows = len(images)
    row_titles = None
    column_titles = group_titles

models_slug = "_".join(name.lower() for name in model_names)
output_dir = Path("media") / "targeted_accentuations"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / (
    f"targeted_accentuations_{seed}_{models_slug}_{METHOD}"
    f"_target_{target_file_tag}"
    f"_alpha_{alpha}_K_{K}_models_{MODEL_LAYOUT}.jpg"
 )

plot_example_grid(
    figure_batch,
    nrow=grid_columns,
    row_titles=row_titles,
    column_titles=column_titles,
    figsize=(3.2 * grid_columns, 3.0 * grid_rows),
    dpi=100,
    fontsize=18,
    # heatmap=True,
    save_path=str(output_path),
)
# plt.close()

# print(f"Saved figure to: {output_path.resolve()}")
# display(Image(filename=str(output_path)))